# Multi-File MIMO Analysis with BalancePy

This notebook demonstrates the new multi-file Multi-Input Multi-Output (MIMO) workflow for analyzing balance data across multiple stimulus-response pairs.

## Problem Solved

Previously, analyzing multiple stimuli and responses required awkward workarounds:
- Loading the same data file multiple times (inefficient)
- Creating separate sr_data objects manually (error-prone)
- No way to combine stimulus from one file with response from another

The new `get_sr_data(configs=...)` and `plot_datacheck_multi()` API solves these issues by:
- Loading multiple files once and extracting all stimulus-response pairs
- Supporting flexible pairing (stimulus and response from different files)
- Visualizing all pairs together for data quality assessment
- Enabling joint fitting across multiple stimulus conditions

## Workflow Overview

```
1. Define configs       → List[(filename, AnaropiaSRDataConfig), ...]
2. Load data           → get_sr_data(configs=...)
3. Visualize           → plot_datacheck_multi(sr_dict)
4. Create models       → [Model1(sr_dict[key1]), Model2(sr_dict[key2]), ...]
5. Fit jointly         → MultiModel(models).fit()
```

## 1. Setup and Imports

In [ ]:
import numpy as np
import balancepy as bp
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("BalancePy version:", bp.__version__ if hasattr(bp, '__version__') else 'dev')

## 2. Single-File Workflow (Backward Compatible)

The original API still works unchanged. This loads multiple stimuli from a single file.

In [ ]:
# Example: Load stimulus data
# (This requires actual data files; using placeholder paths)

# Configure preprocessing (shared across all stimuli/files)
preproc_config = bp.AnaropiaPreprocessingConfig(
    samplingrate_Hz=90,
    resample=True,
    end_time_seconds=260,
    filter_type='lowpass',
    filter_order=4,
    filter_cutoff_Hz=5.0,
    cut_to_cycles=True,
    cycle_start_samples=300,
    cycle_length_samples=2700,
)

# OLD API: Single config, single file
sr_config = bp.AnaropiaSRDataConfig(
    stimulus_column='stim_pitch',  # Single stimulus
    com_config=bp.COM_LEGACY_AP,
)

print("\nOld API signature:")
print("sr_dict = bp.get_sr_data(")
print("    filename='data.csv',")
print("    body_height_m=1.75,")
print("    sr_config=sr_config,")
print("    preproc_config=preproc_config,")
print(")")
print("# Returns: {'stim_pitch': sr_data(...)}")

## 3. Multi-File MIMO Workflow (New!)

The new API accepts a list of (filename, sr_config) tuples, allowing flexible multi-stimulus and multi-response combinations.

In [ ]:
# NEW API: Multiple files with different configs
# Each config specifies a file and its stimulus/response mapping

configs = [
    # File 1: Pitch stimulus → AP response
    (
        'session1.csv',
        bp.AnaropiaSRDataConfig(
            stimulus_column='stim_pitch',
            com_config=bp.COM_LEGACY_AP,  # AP (anterior-posterior) response
        ),
    ),
    # File 2: Roll stimulus → ML response
    (
        'session2.csv',
        bp.AnaropiaSRDataConfig(
            stimulus_column='stim_roll',
            com_config=bp.COM_LEGACY_ML,  # ML (medial-lateral) response
        ),
    ),
    # File 3: Yaw stimulus → response (from same file)
    (
        'session1.csv',  # Can reuse files!
        bp.AnaropiaSRDataConfig(
            stimulus_column='stim_yaw',
            com_config=bp.COM_LEGACY_AP,
        ),
    ),
]

print("New API signature:")
print("sr_dict = bp.get_sr_data(")
print("    body_height_m=1.75,")
print("    preproc_config=preproc_config,  # Shared preprocessing")
print("    configs=[")
print("        ('session1.csv', config1),")
print("        ('session2.csv', config2),")
print("        ('session1.csv', config3),  # Different response from same file")
print("    ]")
print(")")
print("\n# Returns dict with keys: ('session1.csv', 'stim_pitch'), ('session2.csv', 'stim_roll'), ('session1.csv', 'stim_yaw')")
print("# Benefits:")
print("#   - Load each file only once")
print("#   - Flexible stimulus-response pairing")
print("#   - Explicit data flow (no implicit assumptions)")

## 4. Data Loading Example (Pseudocode)

Here's what happens under the hood:

In [ ]:
# Pseudocode showing what get_sr_data does internally:

pseudocode = """
def get_sr_data(body_height_m, preproc_config, configs=None):
    result = {}
    
    for filename, sr_config in configs:
        # Load file once
        raw_data = load_csv(filename)
        
        # Extract RESPONSE once (shared across all stimuli in this config)
        response_raw = extract_response(raw_data, sr_config, body_height_m)
        response_processed = preprocess(response_raw, preproc_config)
        
        # For each STIMULUS in this config:
        for stimulus_col in sr_config.stimulus_column:
            stimulus_raw = extract_stimulus(raw_data, stimulus_col, sr_config)
            stimulus_processed = preprocess(stimulus_raw, preproc_config)
            
            # Create sr_data pair
            sr_data = create_sr_data(
                stimulus=stimulus_processed,
                response=response_processed,  # Shared!
                samplingrate=preproc_config.samplingrate_Hz,
            )
            
            # Store with explicit key
            result[(filename, stimulus_col)] = sr_data
    
    return result
"""

print(pseudocode)
print("\nKey insight: Each FILE is loaded ONCE, response is SHARED within file,")
print("but DIFFERENT files can use different responses!")

## 5. Visualization: plot_datacheck_multi()

The new `plot_datacheck_multi()` function plots all loaded stimulus-response pairs together.

In [ ]:
# Example usage (when you have actual data):
# 
# sr_dict = bp.get_sr_data(
#     body_height_m=1.75,
#     preproc_config=preproc_config,
#     configs=configs,
# )
#
# fig, _ = bp.plot_datacheck_multi(
#     sr_dict,
#     output_dir='results/datacheck_plots/',
#     save=True,
# )
# fig.show()

print("plot_datacheck_multi() features:")
print("  ✓ One row per stimulus-response pair")
print("  ✓ Columns: stim(time), resp(time), stim(freq), resp(freq)")
print("  ✓ Shows mean (solid) + individual cycles (light) if 2D data")
print("  ✓ Frequency domain (FFT magnitude) for both stim and resp")
print("  ✓ Labels show filename:stimulus_col for each row")
print("  ✓ Saves as PNG (datacheck_multi.png)")

## 6. Multi-Model Fitting Workflow

Joint fitting of shared body parameters across multiple stimulus-response pairs.

In [ ]:
# Example: Create models for each stimulus-response pair
# 
# sr_dict = bp.get_sr_data(..., configs=configs)
#
# # Create independent models (each for its own sr_data)
# model_pitch = bp.Peterka18(
#     mass_kg=80,
#     height_m=1.75,
#     data_exp=sr_dict[('session1.csv', 'stim_pitch')],
# )
#
# model_roll = bp.Peterka18(
#     mass_kg=80,
#     height_m=1.75,
#     data_exp=sr_dict[('session2.csv', 'stim_roll')],
# )
#
# model_yaw = bp.Peterka18(
#     mass_kg=80,
#     height_m=1.75,
#     data_exp=sr_dict[('session1.csv', 'stim_yaw')],
# )
#
# # Combine into MultiModel
# multi = bp.MultiModel([model_pitch, model_roll, model_yaw])
#
# # Fit: optimizes shared parameters to minimize error across ALL models
# theta_fit, fit_output = multi.fit()
#
# # Result: One set of fitted parameters explaining all three stimuli!

print("Multi-model fitting advantages:")
print("  ✓ Shared body parameters (mgh, J) optimized across stimuli")
print("  ✓ Different response channels (AP, ML) can have different parameters")
print("  ✓ More data → better constrained fit")
print("  ✓ Tests whether one model explains behavior across conditions")

## 7. Advanced: Flexible Cross-File Pairing

You can even combine stimulus from one file with response from another!

In [ ]:
# Advanced pairing: Stimulus from file A, response from file B
# (Use case: sessions with different equipment or timing)

advanced_configs = [
    # Load pitch stimulus, extract AP response from SAME file
    (
        'session_A.csv',
        bp.AnaropiaSRDataConfig(
            stimulus_column='stim_pitch',
            com_config=bp.COM_LEGACY_AP,
        ),
    ),
    # Load DIFFERENT stimulus, but extract response from DIFFERENT file
    # (Requires manual handling: load session_B, use its response config)
    (
        'session_B.csv',
        bp.AnaropiaSRDataConfig(
            stimulus_column='stim_roll',
            com_config=bp.COM_LEGACY_ML,  # Different response source!
        ),
    ),
]

print("Use case: Cross-session analysis")
print("  - Session A: Good pitch stimulus, poor ML response")
print("  - Session B: Poor pitch stimulus, good ML response")
print("  → Load pitch from A, ML response from B, fit jointly!")
print()
print("Constraint: stimulus_col and response_col must come from SAME file")
print("(Within a single sr_config call)")

## 8. Best Practices

Guidelines for using the multi-file MIMO workflow:

In [ ]:
best_practices = """
1. CONFIGURATION
   - Use ONE AnaropiaSRDataConfig per (file, stimulus-response pair)
   - Preprocessing config is SHARED (same for all files)
   - Different files → different sr_configs for clarity

2. DATA ORGANIZATION
   - Store configs in a list to avoid repetition
   - Name files consistently (e.g., session1.csv, session2.csv)
   - Document which stimulus/response column is in each file

3. VISUALIZATION
   - Always call plot_datacheck_multi() before fitting
   - Check for:
     ✓ Correct stimulus and response signals
     ✓ Data quality (noise, artifacts)
     ✓ Frequency content (SNR, coherence)
     ✓ Cycle consistency

4. FITTING
   - Use MultiModel to fit across multiple stimuli
   - Body parameters (height, mass) should be SAME for all models
   - Stimulus-specific parameters (gains) should have different names
   - Check that error decreases across all models

5. VALIDATION
   - Plot fitted FRF vs experimental FRF for each stimulus
   - Check parameter distributions (are they physically plausible?)
   - Compare to single-stimulus fits (should be similar)
   - Report error/coherence per stimulus
"""

print(best_practices)

## 9. Troubleshooting

Common issues and solutions:

In [ ]:
troubleshooting = """
❌ Problem: "Parameter 'X' not found in MultiModelParameterSet"
   Solution: Check that multimodel_name is set consistently in all models
            Different models must use same param names for shared params

❌ Problem: Fit fails or converges poorly
   Solution: Check plot_datacheck_multi() output:
            - Are all signals present and reasonable?
            - Are frequency ranges appropriate?
            - Try different initial parameter values
            - Consider fitting individual stimuli first, then multi-fit

❌ Problem: Stimulus from one file, response from another breaks
   Solution: Stimulus and response must be from SAME FILE in one sr_config
            Use separate sr_configs for different file combinations

❌ Problem: Data has different sampling rates in different files
   Solution: Specify different preproc_configs? NO: preprocessing is shared
            Instead: resample all files to same rate BEFORE loading
            OR: use separate get_sr_data() calls for different preproc

❌ Problem: plot_datacheck_multi() shows strange plots
   Solution: Check that sr_data objects have correct stimulus/response
            Use simple debugging: print(sr_dict[key].name)
            Verify frequency data computed: print(sr_dict[key].frf.shape)
"""

print(troubleshooting)

## 10. Summary

The new multi-file MIMO workflow provides:

In [ ]:
summary = """
API CHANGES:
  ✓ get_sr_data(filename=..., sr_config=...)  ← OLD API still works
  ✓ get_sr_data(body_height_m=..., configs=[...])  ← NEW multi-file
  ✓ plot_datacheck_multi(sr_dict)  ← NEW visualization

BENEFITS:
  ✓ Load each file once (efficient)
  ✓ Flexible stimulus-response pairing (powerful)
  ✓ Clear data flow (no implicit assumptions)
  ✓ Joint multi-stimulus fitting (better constraints)
  ✓ Comprehensive visualization (data quality)

NEXT STEPS:
  1. Prepare your data files (one per (stimulus, response) condition)
  2. Define AnaropiaSRDataConfig for each (file, signal) pair
  3. Call get_sr_data(body_height_m=..., configs=...)
  4. Visualize with plot_datacheck_multi()
  5. Create models and fit with MultiModel
  6. Validate results across all stimuli
"""

print(summary)

## References

- See `demo_multi_model.ipynb` for MultiModel fitting details
- See `balancepy/anaropia.py` docstrings for `get_sr_data()` and `plot_datacheck_multi()`
- See `balancepy/model_sim/multi_model.py` for MultiModel class documentation